# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 16 · Earlier-game behavior: evidence and leakage-safe features

**Feature research remains open. Round 6 failed; its passing-line treatments are stopped unchanged.**

This notebook constructs learned motion-response histories rather than another geometric feature bank. Same-date outcomes are excluded; evaluation uses a frozen training table. There are no new coordinates, new raw data, or scientific model fits in this notebook. The first charts use your uploaded Round 6 aggregates; new diagnostics require your execution.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round7')
OUT = Path('/home/sagemaker-user/nfl-feature-round7-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract this kit first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=2):
    cmd = [str(PY), str(KIT/'run_round.py'), stage, '--fold', str(fold)]
    env = os.environ.copy()
    env.update(OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
    process = subprocess.Popen(cmd, cwd=str(KIT), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        process.wait(timeout=20)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped with exit {code}. Keep checkpoints and return the generated report; do not change settings.')
def read(name):
    return json.loads((OUT/name).read_text())
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## 1. Review the completed evidence
Absolute scores below belong to the reused diagnostic games, not Kaggle or the historical 0.62 model. History improved the inferior terminal treatment but did not beat the control.

In [ ]:
show(visuals.previous_metrics(KIT), 'round6_metrics')

In [ ]:
show(visuals.previous_intervals(KIT), 'round6_intervals')
show(visuals.previous_error_budget(KIT), 'round6_error_budget')

## 2. Verify preserved controls — do not refit them
Run one offline, bounded preflight. It verifies the reviewed data/source chain and forward-replays the four original control-coordinate models. Do not reinstall packages, repeat data downloads, or change the lock.

In [ ]:
run('preflight')
r = read('preflight.json')
assert r['status'] == 'history_preflight_passed'
print({k:r[k] for k in ['status','parent_models_replayed','old_models_refitted','environment']})

## 3. A 32-play, training-only leakage smoke test
The label interface receives training targets only. Queries on a date are encoded before any outcome from that date is added. Counterfactual tests alter same-day and last-day targets and require invariant earlier features. A player-play contributes once per phase, not once per 10-Hz row.

In [ ]:
run('smoke')
r = read('smoke.json')
assert r['status'] == 'history_smoke_passed'
print({k:r[k] for k in ['plays','training_rows','same_day_poison_test_passed','future_poison_test_passed','new_model_fits']})

## 4. Freeze fold-local historical encoders
Build separate ordered-training and frozen-evaluation matrices for folds 2 and 3. Previously evaluated dates may be legitimate training dates in the later fold, never within their own evaluation fold. Completed fold encoders are reused. No raw input or output CSV is reread. Dispersion measures describe trajectory-bin average residual rates, not uncertainty intervals.

In [ ]:
run('prepare')
r = read('preparation.json')
assert r['status'] == 'history_features_ready'
show(visuals.support_coverage(OUT), 'history_support')
show(visuals.support_counts(OUT), 'history_counts')
print('Feature stage complete. Save this notebook. Next: 17_historical_response_ablation.ipynb.')

## What has—and has not—been established
These checks establish information-time behavior and input support, not predictive value. Cold-start fallback and shrinkage are fixed in advance. Do not tune binning or smoothing after inspecting support. Both first-day training rows and unknown evaluation players are retained. All prior tables remain private.